In [ ]:
# Make sure the notebook can import the modules:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))

In [ ]:
import re
import pandas as pd
import pdfplumber


In [ ]:
from plausibility import run_checks
from aggregate import aggregate
from query import select, distinct

### Load data: 

In [ ]:
ROOT = Path.cwd().parent
df = pd.read_csv(ROOT / "data" / "all_consolidated.csv")

In [ ]:
df.head()

### Check all current scale names: 

In [ ]:
set(df["scale"])

### Look at all current scale names belonging to one scale:

In [ ]:
#set(df[df["scale"].str.contains("CTQ")].scale)
#set(df[df["scale"].str.contains("DERS")].scale)
#set(df[df["scale"].str.contains("DES")].scale)
set(df[df["scale"].str.contains("PSYRATS")].scale)

### Define string patterns and the corresponding desired scale names: 

In [ ]:
scale_string_patterns = ['DES', 'DERS', 'CTQ', 'PSYRATS']

scale_names_dic = {
    'DES': 'DES_T',
    'DERS': 'DERS',
    'CTQ': 'CTQ_SF',
    'PSYRATS': 'PSYRATS'
}

### Add the new scale column and the subscale columns to dataframe:

In [ ]:
def determine_scale_name(names_dictionary, string_patterns, string):
    scale_name = "UNMATCHED"
    for string_pattern in string_patterns:
        
        if string_pattern in string:
            
            scale_name = names_dictionary[string_pattern]
            
    return scale_name
    

In [ ]:
def norm_scale_names(dataframe, names_dictionary, string_patterns):
    scale_names = []

    for index, row in dataframe.iterrows():

        scale_name = determine_scale_name(names_dictionary, string_patterns, row['scale'])
        
        scale_names.append(scale_name)
        
    return scale_names


In [ ]:
# Make list with new uniform scale names: 
scale_names = norm_scale_names(df, scale_names_dic, scale_string_patterns)
print(len(scale_names))
print(df.shape)
print(scale_names[0:3])

In [ ]:
"UNMATCHED" in scale_names       # False = every row matched


In [ ]:
# Rename the current scale column containing the non-uniform obsolete scale names: 
df.rename(columns={"scale": "scale_old"}, inplace=True)

In [ ]:
df.head(3)

In [ ]:
df.columns

In [ ]:
# Add the list containing the uniform scale names as the new scale column: 
df['scale'] = scale_names
df.head()

In [ ]:
df.columns

In [ ]:
df.head(3)

### Define string patterns and the corresponding subscale names:

In [ ]:
set(df["scale_old"])

In [ ]:
# All old obsolete DERS scale names: 
ders_list = list(set(df[df["scale_old"].str.contains("DERS")].scale_old))
ders_list

In [ ]:
# Define DERS subscale names: 
ders_subscales = ['NONACCEPTANCE',
                   'GOALS',
                   'IMPULSE',
                   'AWARENESS',
                   'STRATEGIES',
                   'CLARITY']

In [ ]:
# Attribute sting patterns (to be found in old scale names) to subscale names:
ders_subscale_patterns = {'NONACCEPTANCE': ['nonacceptance', 'Nonacceptance'],
                    'GOALS': ['goals', 'Goals', 'goal', 'Goal'],
                    'IMPULSE': ['impulse', 'Impulse', 'impulsive', 'Impulsive'],
                    'AWARENESS': ['awareness', 'Awareness'],
                    'STRATEGIES': ['strategies', 'Strategies'],
                    'CLARITY': ['clarity', 'Clarity']}


In [ ]:
# Make dictionary attributing obsolete DERS scale names (which include item names) to the corresponding
# subscale names: 
ders_subscale_strings = {}
# Loop through DERS subscales and associated string patterns:
for ders_subscale, subscale_patterns in ders_subscale_patterns.items(): 
    # Make an empty list to collect obsolete scale names associated with the subscale in question:
    subscale_exp_list = []
    # Loop through all of the obsolete DERS scale names:
    for ders_exp in ders_list:
        # if (subscale_patterns[0] in ders_exp) or (subscale_patterns[1] in ders_exp):
        # For each obsolete DERS scale name check if it contains one of the string patterns:
        for pattern in subscale_patterns:
            if pattern in ders_exp:
                # When a matching obsolete DERS scale name is found add it to the list:
                subscale_exp_list.append(ders_exp)
                break
                # break loop once the expression has been attributed to the corresponding subscale
                # to avoid attributing it twice.
    print(subscale_exp_list)
    print('\n')
    ders_subscale_strings[ders_subscale] = subscale_exp_list

In [ ]:
# Dictionary attributing new DERS subscale names to the obsolete old DERS scale names: 
ders_subscale_strings

In [ ]:
ders_subscale_strings['CLARITY']

In [ ]:
# Make a dictionary attributing all new subscale names to the obsolete old scale names, 
# integrating the DERS subscale name dictionary and completing the rest manually:
subscale_names_dic = {
    'none': [
        'DES-T',
         'DES-T_item_1',
         'DES-T_item_2',
         'DES-T_item_3',
         'DES-T_item_4',
         'DES-T_item_5',
         'DES-T_item_6',
         'DES-T_item_7',
         'DES-T_item_8',
         'DERS_total',
         'DERS_Total',
         'CTQ_total',
         'CTQ_MD',
         'CTQ_IU'
    ],
    'AHS': [
        'PSYRATS-AHS_total',
         'PSYRATS-AS_beliefs_re_origin',
         'PSYRATS-AS_control',
         'PSYRATS-AS_disruption',
         'PSYRATS-AS_distress_amount',
         'PSYRATS-AS_distress_intensity',
         'PSYRATS-AS_duration',
         'PSYRATS-AS_frequency',
         'PSYRATS-AS_location',
         'PSYRATS-AS_loudness',
         'PSYRATS-AS_negative_content_amount',
         'PSYRATS-AS_negative_content_degree',
         'PSYRATS-AS_total'
    ],
    'DS': [
        'PSYRATS-DS_conviction',
         'PSYRATS-DS_disruption',
         'PSYRATS-DS_distress_amount',
         'PSYRATS-DS_distress_intensity',
         'PSYRATS-DS_preoccupation_amount',
         'PSYRATS-DS_preoccupation_duration',
         'PSYRATS-DS_total'
    ],
    'NONACCEPTANCE': ders_subscale_strings['NONACCEPTANCE'],
    'GOALS': ders_subscale_strings['GOALS'],
    'IMPULSE': ders_subscale_strings['IMPULSE'],
    'AWARENESS': ders_subscale_strings['AWARENESS'],
    'STRATEGIES': ders_subscale_strings['STRATEGIES'],
    'CLARITY': ders_subscale_strings['CLARITY'],
    'EA': [
        'CTQ-SF_emotional_abuse',
         'CTQ-SF_emotional_abuse_item_14',
         'CTQ-SF_emotional_abuse_item_18',
         'CTQ-SF_emotional_abuse_item_25',
         'CTQ-SF_emotional_abuse_item_3',
         'CTQ-SF_emotional_abuse_item_8',
         'CTQ_EA'
    ],
    'PA': [
        'CTQ-SF_physical_abuse',
         'CTQ-SF_physical_abuse_item_11',
         'CTQ-SF_physical_abuse_item_12',
         'CTQ-SF_physical_abuse_item_15',
         'CTQ-SF_physical_abuse_item_17',
         'CTQ-SF_physical_abuse_item_9', 
         'CTQ_PA'
    ],
    'SA': [
        'CTQ-SF_sexual_abuse',
         'CTQ-SF_sexual_abuse_item_20',
         'CTQ-SF_sexual_abuse_item_21',
         'CTQ-SF_sexual_abuse_item_23',
         'CTQ-SF_sexual_abuse_item_24',
         'CTQ-SF_sexual_abuse_item_27',
         'CTQ_SA'
    ],
    'EN': [
        'CTQ-SF_emotional_neglect',
         'CTQ-SF_emotional_neglect_item_13R',
         'CTQ-SF_emotional_neglect_item_19R',
         'CTQ-SF_emotional_neglect_item_28R',
         'CTQ-SF_emotional_neglect_item_5R',
         'CTQ-SF_emotional_neglect_item_7R',
         'CTQ_EN'
    ],
    'PN': [
        'CTQ-SF_physical_neglect',
         'CTQ-SF_physical_neglect_item_1',
         'CTQ-SF_physical_neglect_item_26R',
         'CTQ-SF_physical_neglect_item_2R',
         'CTQ-SF_physical_neglect_item_4',
         'CTQ-SF_physical_neglect_item_6',
         'CTQ_PN'
    ],
    'MD': [
        'CTQ-SF_validity_item_10R',
         'CTQ-SF_validity_item_16R',
         'CTQ-SF_validity_item_22R',
        'CTQ_MD'
    ],
    'AHS': [
        'PSYRATS-AHS_total',
         'PSYRATS-AS_beliefs_re_origin',
         'PSYRATS-AS_control',
         'PSYRATS-AS_disruption',
         'PSYRATS-AS_distress_amount',
         'PSYRATS-AS_distress_intensity',
         'PSYRATS-AS_duration',
         'PSYRATS-AS_frequency',
         'PSYRATS-AS_location',
         'PSYRATS-AS_loudness',
         'PSYRATS-AS_negative_content_amount',
         'PSYRATS-AS_negative_content_degree',
         'PSYRATS-AS_total'
    ],
    'DS': [
        'PSYRATS-DS_conviction',
         'PSYRATS-DS_disruption',
         'PSYRATS-DS_distress_amount',
         'PSYRATS-DS_distress_intensity',
         'PSYRATS-DS_preoccupation_amount',
         'PSYRATS-DS_preoccupation_duration',
         'PSYRATS-DS_total'       
    ]
}

### Add subscale column: 

In [ ]:
def determine_subscale_name(subscale_names_dictionary, string):
    desired_subscale_name = "UNMATCHED"
    for subscale, strings in subscale_names_dictionary.items():
        if string in strings:
            desired_subscale_name = subscale
    return desired_subscale_name

In [ ]:
def norm_subscale_names(dataframe, subscale_names_dictionary):
    subscale_names = []
    
    for index, row in dataframe.iterrows():

        subscale_name = determine_subscale_name(subscale_names_dictionary, row['scale_old'])
        
        subscale_names.append(subscale_name)

    return subscale_names


In [ ]:
# Define list with new uniform subscale names: 
subscales = norm_subscale_names(df, subscale_names_dic)
print(len(subscales))
print(df.shape)

In [ ]:
"UNMATCHED" in subscales # False = every row matched


In [ ]:
df.head()

In [ ]:
df['subscale'] = subscales
df.columns

In [ ]:
df.head(3)

### Check newly added subscale names:

In [ ]:
set(df.subscale)

In [ ]:
set(df.scale)

In [ ]:
df[df.scale_old=='CTQ_MD'][['scale_old', 'subscale']]

In [ ]:
df[df.scale=='CTQ_SF'][['scale_old', 'subscale']].iloc[3,:]

In [ ]:
df[df.scale=='CTQ_SF'][['scale_old', 'subscale']].iloc[12,:]

In [ ]:
df[df.scale=='CTQ_SF'][['scale_old', 'subscale']].iloc[37,:]

In [ ]:
df[df.scale=='CTQ_SF'][['scale_old', 'subscale']].iloc[150,:]

In [ ]:
df[df.scale=='CTQ_SF'][['scale_old', 'subscale']].iloc[157,:]

In [ ]:
df[df.scale=='DERS'][['scale_old', 'subscale']].iloc[0,:]

In [ ]:
df[df.scale=='DES_T'][['scale_old', 'subscale']].iloc[59,:]

In [ ]:
df[df.scale=='PSYRATS'][['scale_old', 'subscale']].iloc[34, :]

### Identify old scale names referring to individual items: 

In [ ]:
# Define dictionary to store item names by scale:
individual_items = {}

#### Get PSYRATS item names:

In [ ]:
len(set(df[df.scale=='PSYRATS'].scale_old))

In [ ]:
psyrats_old_scale_names = list(set(df[df.scale=='PSYRATS'].scale_old))

psyrats_item_names = [name for name in psyrats_old_scale_names if 'total' not in name]
print(len(psyrats_item_names))
psyrats_item_names

In [ ]:
individual_items['PSYRATS'] = psyrats_item_names
len(individual_items['PSYRATS'])

#### Get CTQ_SF items:

In [ ]:
ctq_old_scale_names = list(set(df[df.scale=='CTQ_SF'].scale_old))

CTQ_items = [name for name in ctq_old_scale_names if (any(c.isdigit() for c in name))]

print(len(CTQ_items))
CTQ_items

In [ ]:
CTQ_validity_items = [name for name in CTQ_items if 'validity' in name]
CTQ_validity_items 

In [ ]:
individual_items['CTQ_SF'] = CTQ_items
len(individual_items['CTQ_SF'])

#### Get DES_T items: 

In [ ]:
dest_old_scale_names = list(set(df[df.scale=='DES_T'].scale_old))

DES_T_items = [name for name in dest_old_scale_names if any(c.isdigit() for c in name)]
print(len(DES_T_items))
DES_T_items

In [ ]:
individual_items['DES_T'] = DES_T_items
len(individual_items['DES_T'])

### Add record type and item name column to dataframe: 

In [ ]:
all_items = []
for key, items in individual_items.items():
    print(key)
    
    all_items.extend(items)

In [ ]:
len(all_items)

In [ ]:
for key, items in individual_items.items():
    print(key)
    print(len(items))

In [ ]:
# Make item_names list: 
item_names = []

for index, row in df.iterrows():

    if row.scale_old in all_items:
        item_names.append(row.scale_old)
    else:
        item_names.append("none")

In [ ]:
print(df.shape)
print(len(item_names))

In [ ]:
# Add item names column: 
df['item_name'] = item_names

In [ ]:
df.columns

In [ ]:
df.head()

In [ ]:
# Make record type list: 
record_type = []
for index, row in df.iterrows():
    
    if (row.item_name == "none") and (row.subscale == "none") and (row.scale_old != 'CTQ_IU'): 
        # CTQ_IU is not part of our analysis, so we are not interested.
        record_type.append('total')
    elif (row.item_name == "none") and (row.subscale != "none"):
        record_type.append('subscale')
    elif (row.item_name !=  "none"):
        record_type.append("item")
    else:
        print(row.scale_old, row.scale, row.subscale, row.item_name)
        record_type.append("unknown")


In [ ]:
print(len(record_type))
print(df.shape)

In [ ]:
# Add record type column: 
df['record_type'] = record_type


In [ ]:
df.head()

### Have a look at the data: 

In [ ]:
df[df.record_type == 'total'].head()

In [ ]:
# Have a look at the old scale names of records referring to total scales: 
set(df[df.record_type == 'total'].scale_old)

In [ ]:
# Have a look at the records of record type subscale: 
set(df[df.record_type == 'subscale'].scale_old)

In [ ]:
# All records of record type 'subscale' must have a subscale name:
set(df[df.record_type == 'subscale'].subscale)

In [ ]:
# Have a look at the old scale names of the item records: 
set(df[df.record_type == 'item'].scale_old)

In [ ]:
len(set(df[df.record_type == 'item'].scale_old))

In [ ]:
# Record type 'item' must not have 'none' as an item name, hence
# the two dataframe slices must have the same dimensions:
print(df[df.item_name != 'none'].shape)
print(df[df.record_type == 'item'].shape)


In [ ]:
df[df.item_name == 'none'].scale_old

In [ ]:
print(df[df.item_name == 'none'].shape)
print(df[df.record_type == 'item'].shape)
print(df[df.record_type == 'total'].shape)
print(df[df.record_type == 'subscale'].shape)

In [ ]:
# Every record must have a record type or unknown so they should sum up
# to the total number of records: 
(df[df.record_type == 'item'].shape[0] + 
df[df.record_type == 'total'].shape[0] + 
df[df.record_type == 'subscale'].shape[0] +
df[df.record_type=='unknown'].shape[0])



In [ ]:
# Every record either refers to an item or not, hence the number of none-item records plus
# the number of item records equals the total number of records: 
print(df[df.item_name == 'none'].shape[0] + df[df.record_type == 'item'].shape[0])
print(df.shape)

In [ ]:
# Records not referring to individual items should not have
# an item name, hence the following slice should have a dimension
# of 0 rows:
df[df.record_type != 'item'][df.item_name != 'none']

### Add REDCap item numbers for records referring to individual DES-T and PSYRATS items: 

#### I shall not use the CTQ records referring to individual items as the respective publication (Garcia-Fernandez et al. 2024) uses item numbering inconsistent with ours and fails to describe the individual items.

In [ ]:
set(df[df.record_type=='item'].scale)

### Have a look at records referring to individual items: 

In [ ]:
df_items = df[df.record_type=='item'].copy()


In [ ]:
set(df_items[df_items.scale=='DES_T'].source_file)

In [ ]:
# Define which DES-T item names correspond to which REDCap item numbers
# (by visual inspection of the publication by Spitzer et al. 2014): 
dest_item_numbering_spitzer_redcap = {
    'DES-T_item_1': 1,
    'DES-T_item_2': 6,
    'DES-T_item_3': 3,
    'DES-T_item_4': 8,
    'DES-T_item_5': 4,
    'DES-T_item_6': 2,
    'DES-T_item_7': 5,
    'DES-T_item_8': 7
}

In [ ]:
set(df_items[df_items.scale=='PSYRATS'].source_file)

In [ ]:
set(df_items[df_items.scale=='PSYRATS'].item_name)

In [ ]:
# Define which PSYRATS item names correspond to which REDCap item numbers
# (by visual inspection): 

psyrats_item_numbering_haddock_redcap = {

        'PSYRATS-AS_frequency': 1,
        'PSYRATS-AS_duration': 2,
        'PSYRATS-AS_location': 3,
        'PSYRATS-AS_loudness': 4, 
        'PSYRATS-AS_beliefs_re_origin': 5,
        'PSYRATS-AS_negative_content_amount': 6,
        'PSYRATS-AS_negative_content_degree': 7,
        'PSYRATS-AS_distress_amount': 8,
        'PSYRATS-AS_distress_intensity': 9,
        'PSYRATS-AS_disruption': 10,
        'PSYRATS-AS_control': 11,
        'PSYRATS-DS_preoccupation_amount': 1,
        'PSYRATS-DS_preoccupation_duration': 2,
        'PSYRATS-DS_conviction': 3,
        'PSYRATS-DS_distress_amount': 4,
        'PSYRATS-DS_distress_intensity': 5,
        'PSYRATS-DS_disruption': 6
    
}

In [ ]:
# Loop through dataframe and make a redcap_item_numbers list: 
redcap_item_numbers = []
for index, row in df.iterrows():
    
    if (row.record_type == 'item') and (row.scale == 'DES_T'):
        redcap_item_numbers.append(dest_item_numbering_spitzer_redcap[row.scale_old])
    elif (row.record_type == 'item') and (row.scale == 'PSYRATS'):
        redcap_item_numbers.append(psyrats_item_numbering_haddock_redcap[row.scale_old])
    else:
        redcap_item_numbers.append('none')
    #break

In [ ]:
len(redcap_item_numbers)

In [ ]:
# Add redcap_item_number column to dataframe: 
df['redcap_item_number'] = redcap_item_numbers

In [ ]:
df_items = df[df.record_type=='item'].copy()

In [ ]:
df_items[df_items.scale=='DES_T'][['scale', 'scale_old', 'item_name', 'redcap_item_number']]

In [ ]:
df_items[df_items.scale=='PSYRATS'][['scale', 'scale_old', 'item_name', 'redcap_item_number']]

### Fill in “whole_sample“ values in the subsample column for records not referring to a subsample:


In [ ]:
df.head()

In [ ]:
# Have a look at the value set in the subsample column: 
set(df.subsample)

In [ ]:
# Check if 'whole_sample' is already being used: 
(df == "whole_sample").any().any()

In [ ]:
# Check how many missing values: 
df["subsample"].isna().sum()

In [ ]:
# Add 'whole_sample' value to subsample column where the record 
# does not refer to a subsample:
df.loc[df["subsample"].isna(), "subsample"] = "whole_sample"

In [ ]:
# Check how many missing values again after adding the 'whole_sample' value
# (there should be no missing values:
df["subsample"].isna().sum()

In [ ]:
# Check how many 'whole_sample' values
# (should equal the number of missing values before): 
sum(df["subsample"] == "whole_sample")

### Rename the sample column:

In [ ]:
df.rename(columns={'sample': 'sample_type'}, inplace=True)

### Add scoring_rule column:
#### Contains scoring rules where known, and "unverified" where scoring rules are unknown. Values are added to the data frame based on literature research as summarized in SummaryScoringConcise (state as of 17th of August, 2026). 

In [ ]:
# Add new column to be filled with values. Default value is "unverified“ 
# as I only
df["scoring_rule"] = "unverified"

In [ ]:
df.columns

In [ ]:
df.head(3)

In [ ]:
set(df.source_file)

In [ ]:
non_item_bools = df.record_type != 'item'
publication_bools = df.source_file == 'CTQ_SF_GarciaFernandez_et_al_2024_long.csv'
df.loc[
    non_item_bools & 
    publication_bools, 
    'scoring_rule'
    ].shape

In [ ]:
set(df.loc[
    non_item_bools & 
    publication_bools
    ].record_type)

In [ ]:

df.loc[
    non_item_bools & 
    publication_bools, 
    'scoring_rule'
    ] = 'raw_sum'

In [ ]:
df[df.scoring_rule=='raw_sum'].shape

In [ ]:
set(df.source_file)

In [ ]:
non_item_bools = df.record_type != 'item'
publication_bools = df.source_file == 'CTQ_SF_Hagborg_et_al_2022_long.csv'
set(df.loc[
    non_item_bools & 
    publication_bools, 
    ('scale', 
    'subscale', 
    'record_type')
    ].record_type)

In [ ]:
non_item_bools = df.record_type.isin({'subscale', 'total'})
publication_bools = df.source_file == 'CTQ_SF_Hagborg_et_al_2022_long.csv'
df.loc[
    non_item_bools & 
    publication_bools, 
    'scoring_rule'
    ].shape

In [ ]:
df.loc[
    non_item_bools & 
    publication_bools, 
    'scoring_rule'
    ] = 'raw_sum'

In [ ]:
df[df.scoring_rule=='raw_sum'].shape

In [ ]:
set(df.source_file)

In [ ]:
non_item_bools = df.record_type != 'item'
publication_bools = df.source_file == 'DERS_Giromini_et_al_2012_long.csv'
set(df.loc[
    non_item_bools & 
    publication_bools, 
    ('scale', 
    'subscale', 
    'record_type')
    ].record_type)

In [ ]:
non_item_bools = df.record_type.isin({'subscale', 'total'})
publication_bools = df.source_file == 'DERS_Giromini_et_al_2012_long.csv'
df.loc[
    non_item_bools & 
    publication_bools, 
    'scoring_rule'
    ].shape

In [ ]:
df.loc[
    non_item_bools & 
    publication_bools, 
    'scoring_rule'
    ] = 'raw_sum'

In [ ]:
df[df.scoring_rule=='raw_sum'].shape

In [ ]:
non_item_bools = df.record_type != 'item'
publication_bools = df.source_file == 'DERS_GratzRoemer_2004_long.csv'
set(df.loc[
    non_item_bools & 
    publication_bools, 
    ('scale', 
    'subscale', 
    'record_type')
    ].record_type)

In [ ]:
non_item_bools = df.record_type.isin({'subscale', 'total'})
publication_bools = df.source_file == 'DERS_GratzRoemer_2004_long.csv'
df.loc[
    non_item_bools & 
    publication_bools, 
    'scoring_rule'
    ].shape

In [ ]:
df.loc[
    non_item_bools & 
    publication_bools, 
    'scoring_rule'
    ] = 'raw_sum'

In [ ]:
df[df.scoring_rule=='raw_sum'].shape

In [ ]:
non_item_bools = df.record_type != 'item'
publication_bools = df.source_file == 'DERS_Neumann_et_al_2010_long.csv'
set(df.loc[
    non_item_bools & 
    publication_bools, 
    ('scale', 
    'subscale', 
    'record_type')
    ].record_type)

In [ ]:
non_item_bools = df.record_type.isin({'subscale', 'total'})
publication_bools = df.source_file == 'DERS_Neumann_et_al_2010_long.csv'
df.loc[
    non_item_bools & 
    publication_bools, 
    'scoring_rule'
    ].shape

In [ ]:
df.loc[
    non_item_bools & 
    publication_bools, 
    'scoring_rule'
    ] = 'raw_sum'

In [ ]:
df[df.scoring_rule=='raw_sum'].shape

In [ ]:
non_item_bools = df.record_type != 'item'
publication_bools = df.source_file == 'DES-T_Giesbrecht_et_al_2007_long.csv'
set(df.loc[
    non_item_bools & 
    publication_bools, 
    ('scale', 
    'subscale', 
    'record_type')
    ].record_type)

In [ ]:
non_item_bools = df.record_type.isin({'subscale', 'total'})
publication_bools = df.source_file == 'DES-T_Giesbrecht_et_al_2007_long.csv'
df.loc[
    non_item_bools & 
    publication_bools, 
    'scoring_rule'
    ].shape

In [ ]:
df.loc[
    non_item_bools & 
    publication_bools, 
    'scoring_rule'
    ] = 'arithmetic_mean'

In [ ]:
df[df.scoring_rule=='arithmetic_mean'].shape

In [ ]:
non_item_bools = df.record_type != 'item'
publication_bools = df.source_file == 'DES-T_Levin_et_al_2003_long.csv'
set(df.loc[
    non_item_bools & 
    publication_bools, 
    ('scale', 
    'subscale', 
    'record_type')
    ].record_type)

In [ ]:
non_item_bools = df.record_type.isin({'subscale', 'total'})
publication_bools = df.source_file == 'DES-T_Levin_et_al_2003_long.csv'
df.loc[
    non_item_bools & 
    publication_bools, 
    'scoring_rule'
    ].shape

In [ ]:
df.loc[
    non_item_bools & 
    publication_bools, 
    'scoring_rule'
    ] = 'arithmetic_mean'

In [ ]:
df[df.scoring_rule=='arithmetic_mean'].shape

In [ ]:
non_item_bools = df.record_type != 'item'
publication_bools = df.source_file == 'DES-T_Modestin_et_al_2004_long.csv'
set(df.loc[
    non_item_bools & 
    publication_bools, 
    ('scale', 
    'subscale', 
    'record_type')
    ].record_type)

In [ ]:
non_item_bools = df.record_type.isin({'subscale', 'total'})
publication_bools = df.source_file == 'DES-T_Modestin_et_al_2004_long.csv'
df.loc[
    non_item_bools & 
    publication_bools, 
    'scoring_rule'
    ].shape

In [ ]:
df.loc[
    non_item_bools & 
    publication_bools
    ]

In [ ]:
df.loc[
    non_item_bools & 
    publication_bools, 
    'scoring_rule'
    ] = 'arithmetic_mean'

In [ ]:
df[df.scoring_rule=='arithmetic_mean'].shape

In [ ]:
non_item_bools = df.record_type != 'item'
publication_bools = df.source_file == 'DES-T_Spitzer_et_al_2014_long.csv'
set(df.loc[
    non_item_bools & 
    publication_bools, 
    ('scale', 
    'subscale', 
    'record_type')
    ].record_type)

In [ ]:
non_item_bools = df.record_type.isin({'subscale', 'total'})
publication_bools = df.source_file == 'DES-T_Spitzer_et_al_2014_long.csv'
df.loc[
    non_item_bools & 
    publication_bools, 
    'scoring_rule'
    ].shape

In [ ]:
df.loc[
    non_item_bools & 
    publication_bools, 
    'scoring_rule'
    ] = 'arithmetic_mean'

In [ ]:
df[df.scoring_rule=='arithmetic_mean'].shape

In [ ]:
non_item_bools = df.record_type != 'item'
publication_bools = df.source_file == 'PSYRATS_Woodward_et_al_2014_long.csv'
set(df.loc[
    non_item_bools & 
    publication_bools, 
    ('scale', 
    'subscale', 
    'record_type')
    ].record_type)

In [ ]:
non_item_bools = df.record_type.isin({'subscale', 'total'})
publication_bools = df.source_file == 'PSYRATS_Woodward_et_al_2014_long.csv'
df.loc[
    non_item_bools & 
    publication_bools, 
    'scoring_rule'
    ].shape

In [ ]:
df.loc[
    non_item_bools & 
    publication_bools, 
    'scoring_rule'
    ] = 'raw_sum'

In [ ]:
df[df.scoring_rule=='raw_sum'].shape

### Add item_score_reversed column 

#### Get all the reversed scored items from Garcia-Fernandez first, the rest is labelled as 'none' if it is not an item score, and 'no' (not reversed) as a default as the rest of the reversed items will be determined afterwards.

In [ ]:
item_bools = df.record_type == 'item'
publication_bools = df.publication == 'Garcia-Fernandez et al. 2024'
GF_CTQ_items = df[item_bools & publication_bools].copy()

In [ ]:
#item_bools = df.record_type == 'item'
source_file_bools = df.source_file == 'CTQ_SF_GarciaFernandez_et_al_2024_items_long.csv'
GF_CTQ_items_2 = df[source_file_bools].copy()

In [ ]:
print(GF_CTQ_items.shape)
print(GF_CTQ_items_2.shape)
print(set(GF_CTQ_items.record_type))
print(set(GF_CTQ_items_2.record_type))

In [ ]:
set(GF_CTQ_items_2.item_name)

In [ ]:
df.shape

In [ ]:
df['item_score_reversed'] = 'none' # default for non-item rows

In [ ]:
df.loc[df.record_type == "item"].shape

In [ ]:
df['item_score_reversed'] = 'none' # default for non-item rows
df.loc[df.record_type == "item", "item_score_reversed"] = "no"  #items: regular unless...
print(df.loc[df.item_score_reversed == "no"].shape)
print(df.loc[df.item_score_reversed == "none"].shape)


In [ ]:
df.shape[0] == (
    df.loc[df.item_score_reversed == "no"].shape[0] + 
    df.loc[df.item_score_reversed == "none"].shape[0]
)



In [ ]:
df.loc[df.scale_old.str.endswith("R"), "item_score_reversed"] = "yes"  # ...name ends in R


In [ ]:
print(df.loc[df.item_score_reversed == "yes"].shape)

In [ ]:
df.shape[0] == (
    df.loc[df.item_score_reversed == "no"].shape[0] + 
    df.loc[df.item_score_reversed == "none"].shape[0] +
    df.loc[df.item_score_reversed == "yes"].shape[0]
)

In [ ]:
df.loc[df.item_score_reversed == "no"].head()

In [ ]:
set(df.loc[df.item_score_reversed == "no"].publication)

#### Get reversed scored items from Haddock et al. 1999

In [ ]:
item_bools = df.record_type == 'item'
publication_bools = df.publication == 'Haddock et al. 1999'
set(df[item_bools & publication_bools].item_name)

#### => Leave unchanged (not reverse scored) as PSYRATS does not have reverse scored items:

#### Get reversed scored items from Spitzer et al. 2014

In [ ]:
item_bools = df.record_type == 'item'
publication_bools = df.publication == 'Spitzer et al. 2014'
set(df[item_bools & publication_bools].item_name)

#### => Leave unchanged (not reverse scored) as DES-T does not have reverse scored items:

### Find instances where 'whole_sample' values are redundant, i.e. represent a sample that could also be constructed by aggregating subsamples in the dataframe:

In [ ]:
df[df.subsample!="whole_sample"].head()

In [ ]:
df[df.subsample!="whole_sample"].shape

In [ ]:
set(df[df.subsample!="whole_sample"].scale)

In [ ]:
# Identify publications with subsamples:
subsample_pub_list = list(set(df[df.subsample!="whole_sample"].source_file))
subsample_pub_list

In [ ]:
# Identify publications with whole_sample and subsample records and 
# the respective records:
# Dictionary to store the data of interest: 
pubs_whole_and_subsamples = {}
# Loop through publications with subsamples: 
for subsample_pub in subsample_pub_list:
    source_file_bools = df.source_file==subsample_pub
    subsample_bools = df.subsample=='whole_sample'
    # Check if the publication also has whole_sample records: 
    num_rows = df[(source_file_bools) & (subsample_bools)].shape[0]
    print(num_rows)
    # If the data from the publication turns out to also have whole_sample records... 
    if num_rows > 0:
        # ...add the data to the dictionary:
        print('yes')
        pubs_whole_and_subsamples[subsample_pub] = df[(source_file_bools)]
    else:
        continue

In [ ]:
pubs_whole_and_subsamples['DES-T_Modestin_et_al_2004_long.csv'].columns

In [ ]:
for key, data_from_pub in pubs_whole_and_subsamples.items():
    print('\n')
    print(key)
    # Check if combinations have unique sample size:
    sample_sizes = data_from_pub.groupby(['scale_old','data_type','sample_type','subsample'])['sample_size'].nunique().max()
    if sample_sizes > 1:
        print('Multiple sample sizes!')
        break
    cols = ['scale_old', 'data_type', 'sample_type', 'subsample', 'sample_size']
    unique_combs = data_from_pub[cols].drop_duplicates()
    if 'mean' in set(unique_combs.data_type):
        unique_combs_means = unique_combs[data_from_pub.data_type=='mean']
    else:
        print('no mean')
        print(unique_combs.columns)
        break
    subsample_rows = unique_combs_means[unique_combs_means.subsample!='whole_sample']
    print('\n')
    print(subsample_rows)
    print(subsample_rows.groupby(['scale_old', 'sample_type'])['sample_size'].sum())
    whole_sample_rows = unique_combs_means[unique_combs_means.subsample=='whole_sample']
    print('\n')
    print(whole_sample_rows)
    

#### Conclusion about the redundant aggregates: 
The records with 'whole_sample' in the subsample column from:

- DES-T_Modestin_et_al_2004_long.csv
- DERS_Giromini_et_al_2012_long.csv
- DERS_Neumann_et_al_2010_long.csv

are redundant aggregates because they could be obtained by aggregating over the respective subsamples.


### Add redundancy flag column for the redundant aggregates

In [ ]:
df['redundant_aggregate'] = False

#### Add the True values for redundant_aggregate records of DES-T_Modestin_et_al_2004_long.csv:

In [ ]:
source_file_bools = (df.source_file=='DES-T_Modestin_et_al_2004_long.csv')
subsample_bools = (df.subsample=='whole_sample')
df[source_file_bools & subsample_bools]

In [ ]:
df[source_file_bools & subsample_bools].shape

In [ ]:
df.loc[source_file_bools & subsample_bools, 'redundant_aggregate'] = True

In [ ]:
df[df.redundant_aggregate==True].shape

In [ ]:
df[df.redundant_aggregate==True]

#### Add the True values for redundant_aggregate records of DERS_Giromini_et_al_2012_long.csv:

In [ ]:
source_file_bools = (df.source_file=='DERS_Giromini_et_al_2012_long.csv')
subsample_bools = (df.subsample=='whole_sample')
df[source_file_bools & subsample_bools]

In [ ]:
df[source_file_bools & subsample_bools].shape

In [ ]:
df.loc[source_file_bools & subsample_bools, 'redundant_aggregate'] = True

In [ ]:
df[df.redundant_aggregate==True].shape

#### Add the True values for redundant_aggregate records of DERS_Neumann_et_al_2010_long.csv:

In [ ]:
source_file_bools = (df.source_file=='DERS_Neumann_et_al_2010_long.csv')
subsample_bools = (df.subsample=='whole_sample')
df[source_file_bools & subsample_bools]

In [ ]:
df[source_file_bools & subsample_bools].shape

In [ ]:
df.loc[source_file_bools & subsample_bools, 'redundant_aggregate'] = True

In [ ]:
df[df.redundant_aggregate==True].shape

In [ ]:
import os
os.getcwd()

In [ ]:
data_corr_path = ROOT / "data" / "all_consolidated_corr.csv"
data_corr_path

In [ ]:
df.to_csv(data_corr_path)

In [ ]:
for key, data_from_pub in pubs_whole_and_subsamples.items():
    print('\n')
    print(key)

In [ ]:
unique_combs_means

In [ ]:
df_modestin_et_al = pubs_whole_and_subsamples['DES-T_Modestin_et_al_2004_long.csv']
df_modestin_et_al.shape

In [ ]:
# Check if combinations have unique sample size:
df_modestin_et_al.groupby(['scale_old','data_type','sample_type','subsample'])['sample_size'].nunique().max()


In [ ]:
cols = ['scale_old', 'data_type', 'sample_type', 'subsample', 'sample_size']
df_modestin_et_al[cols].drop_duplicates()

In [ ]:
df_modestin_et_al_means = df_modestin_et_al[df_modestin_et_al.data_type=='mean']
df_modestin_et_al_means

In [ ]:
df_modestin_et_al_means_whole_sample = df_modestin_et_al_means[df_modestin_et_al_means.subsample=='whole_sample']
df_modestin_et_al_means_whole_sample

In [ ]:
df_modestin_et_al_means_whole_sample_grps = df_modestin_et_al_means_whole_sample.groupby('sample_type')['sample_size'].sum()
df_modestin_et_al_means_whole_sample_grps

In [ ]:
df_modestin_et_al_means_subsample = df_modestin_et_al_means[df_modestin_et_al_means.subsample!='whole_sample']
df_modestin_et_al_means_subsample

In [ ]:
df_modestin_et_al_means_subsample_grps = df_modestin_et_al_means_subsample.groupby('sample_type')['sample_size'].sum()
df_modestin_et_al_means_subsample_grps

In [ ]:
df.groupby('sample_type')['sample_size'].sum()

In [ ]:
type(df_modestin_et_al_means_whole_sample_grps)

In [ ]:
df_modestin_et_al_means_whole_sample_grps['healthy_controls']

In [ ]:
df_modestin_et_al_means_whole_sample_grps.index

In [ ]:
indices = df_modestin_et_al_means_whole_sample_grps.index
indices

In [ ]:
check_bools = []
for index_value in indices:
    print(index_value)
    whole_sample_size = df_modestin_et_al_means_whole_sample_grps[index_value]
    subsample_size = df_modestin_et_al_means_subsample_grps[index_value]
    print(whole_sample_size == subsample_size)
    check_bools.append(whole_sample_size == subsample_size)

    

In [ ]:
sum(check_bools) == len(check_bools)

In [ ]:
source_files_data = set(df.source_file)
source_files_data

In [ ]:
len(set(df.source_file))

In [ ]:
os.getcwd()

In [ ]:
raw_pdfs_path = ROOT / 'data' / 'raw_pdfs'

In [ ]:
raw_pdfs_list = os.listdir(raw_pdfs_path)
raw_pdfs_list


In [ ]:
raw_pdfs_list.sort()
raw_pdfs_list

In [ ]:
len(raw_pdfs_list)

In [ ]:
raw_pdfs = set(raw_pdfs_list)

In [ ]:
intersection = raw_pdfs & source_files_data
intersection

In [ ]:
df.columns

In [ ]:
df[(source_file_bools) & (subsample_bools)&(data_type_bools)].shape

In [ ]:
source_file_bools = df.source_file=='CTQ_SF_Hagborg_et_al_2022_long.csv'
subsample_bools = df.subsample!='whole_sample'
data_type_bools = df.data_type=='mean'

df[(source_file_bools) & (subsample_bools)&(data_type_bools)]

In [ ]:
set(df.redcap_item_number)

In [ ]:
set(df.item_score_reversed)

In [ ]:
set(df.source_file)

In [ ]:
set(df.publication)